# Implied Volatility vs Historical Volatility EDA
This notebook calculates **Annualized** Black-Scholes Implied Volatility (IV) and Historical Volatility (HV) for VELVETFRUIT_EXTRACT options, allowing you to compare them over different rolling windows.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

# ---------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------
DATA_PATH = '../day_0.csv'
TTE_BASE_DAYS = 5
RISK_FREE_RATE = 0.0

# We will analyze these strike prices
STRIKES = [5000, 5100, 5200] 

# Historical Volatility Windows (N ticks)
WINDOWS = [50, 100, 300]

In [ ]:
# ---------------------------------------------------------
# 2. Black-Scholes Bisection Solver
# ---------------------------------------------------------
def norm_cdf(x):
    return (1.0 + math.erf(x / math.sqrt(2.0))) / 2.0

def bs_call_price(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0:
        return max(0.0, S - K)
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return S * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)

def implied_vol_bisection(market_price, S, K, T, r):
    if T <= 0 or market_price <= 0:
        return np.nan
    intrinsic = max(0.0, S - K)
    if market_price <= intrinsic:
        return 0.0
        
    low, high = 1e-4, 10.0
    if bs_call_price(S, K, T, r, high) < market_price:
        return np.nan
        
    for _ in range(100):
        mid = (low + high) / 2.0
        price = bs_call_price(S, K, T, r, mid)
        if price < market_price:
            low = mid
        else:
            high = mid
            
    return (low + high) / 2.0

In [ ]:
# ---------------------------------------------------------
# 3. Load & Preprocess Data
# ---------------------------------------------------------
try:
    df = pd.read_csv(DATA_PATH, sep=';')
except FileNotFoundError:
    print(f"Could not find {DATA_PATH}. Please ensure your Day 0 CSV is in the same directory.")
    df = pd.DataFrame()

if not df.empty:
    # Extract Spot Price series
    spot_df = df[df['product'] == 'VELVETFRUIT_EXTRACT'].copy()
    spot_df.set_index('timestamp', inplace=True)
    spot_df.sort_index(inplace=True)
    
    # Calculate Historical Volatility (Standard Deviation of returns)
    # We'll calculate the log returns of the spot mid_price
    spot_df['log_ret'] = np.log(spot_df['mid_price'] / spot_df['mid_price'].shift(1))
    
    # Calculate HV for the different windows
    for w in WINDOWS:
        # The standard deviation of the log returns over window w (per tick)
        # To annualize: 10,000 ticks per day * 365 days per year = 3,650,000 ticks per year
        spot_df[f'HV_{w}'] = spot_df['log_ret'].rolling(window=w).std() * math.sqrt(3_650_000)

    # Dictionary to hold the option data
    options_data = {}
    
    for strike in STRIKES:
        opt_product = f'VEV_{strike}'
        opt_df = df[df['product'] == opt_product].copy()
        opt_df.set_index('timestamp', inplace=True)
        opt_df.sort_index(inplace=True)
        
        # Align with spot
        merged = opt_df[['mid_price', 'day']].join(spot_df[['mid_price'] + [f'HV_{w}' for w in WINDOWS]], rsuffix='_spot')
        
        ivs = []
        for ts, row in merged.iterrows():
            spot = row['mid_price_spot']
            prem = row['mid_price']
            day = row['day']
            
            if pd.isna(spot) or pd.isna(prem):
                ivs.append(np.nan)
                continue
                
            # Calculate TTE in days, linearly reducing across the day
            tte = TTE_BASE_DAYS - day - (ts / 1_000_000)
            T = tte / 365.0 # Annualized perspective
            
            iv = implied_vol_bisection(prem, spot, strike, T, RISK_FREE_RATE)
            ivs.append(iv)
            
        merged['IV'] = ivs
        options_data[strike] = merged
        
    print("Data processing complete!")

In [ ]:
# ---------------------------------------------------------
# 4. Plotting (Zoomable Plotly Time Series)
# ---------------------------------------------------------
if not df.empty:
    # Create a subplot for each strike
    fig = make_subplots(rows=len(STRIKES), cols=1, 
                        subplot_titles=[f'VEV_{strike}: Implied Vol vs Historical Vol (Time Series)' for strike in STRIKES],
                        shared_xaxes=True,
                        vertical_spacing=0.05)
    
    colors = ['#f59e0b', '#3b82f6', '#10b981', '#ec4899']
    
    for i, strike in enumerate(STRIKES):
        row = i + 1
        merged = options_data[strike]
        
        # Plot IV
        fig.add_trace(go.Scatter(
            x=merged.index, 
            y=merged['IV'],
            mode='lines',
            name=f'IV (VEV_{strike})',
            line=dict(color='#8b5cf6', width=2),
            legendgroup=f'group_{strike}'
        ), row=row, col=1)
        
        # Plot HV for different windows
        for j, w in enumerate(WINDOWS):
            fig.add_trace(go.Scatter(
                x=merged.index, 
                y=merged[f'HV_{w}'],
                mode='lines',
                name=f'HV (N={w})',
                line=dict(color=colors[j % len(colors)], width=1, dash='dot'),
                legendgroup=f'group_{strike}'
            ), row=row, col=1)

    fig.update_layout(
        height=300 * len(STRIKES),
        title_text="Implied Volatility vs Historical Volatility (Time Series)",
        hovermode="x unified",
        template="plotly_dark"
    )
    
    # Add Y-axis titles
    for i in range(len(STRIKES)):
        fig.update_yaxes(title_text="Volatility (Annualized)", tickformat=".1%", row=i+1, col=1)
        
    fig.update_xaxes(title_text="Timestamp", row=len(STRIKES), col=1)

    fig.show()
else:
    print("No data to plot.")

In [ ]:
# ---------------------------------------------------------
# 5. Scatter Plot: IV vs Realized Volatility (RV / HV)
# ---------------------------------------------------------
if not df.empty:
    # We will create a scatter plot with RV on the X-axis and IV on the Y-axis
    # For simplicity, we use the intermediate window (e.g., N=100) as our Realized Vol benchmark.
    target_w = WINDOWS[1] if len(WINDOWS) > 1 else WINDOWS[0]
    rv_col = f'HV_{target_w}'
    
    cols = 2
    rows = math.ceil(len(STRIKES) / cols)
    
    fig2 = make_subplots(rows=rows, cols=cols, 
                         subplot_titles=[f'VEV_{strike}' for strike in STRIKES],
                         vertical_spacing=0.1, 
                         horizontal_spacing=0.05)
    
    for i, strike in enumerate(STRIKES):
        row = (i // cols) + 1
        col = (i % cols) + 1
        merged = options_data[strike].dropna(subset=['IV', rv_col])
        
        # Plot Scatter
        fig2.add_trace(go.Scatter(
            x=merged[rv_col], 
            y=merged['IV'],
            mode='markers',
            marker=dict(
                color=merged.index, 
                colorscale='Viridis', 
                showscale=(i==0), 
                colorbar=dict(title="Timestamp", x=1.05) if i==0 else None,
                size=4,
                opacity=0.7
            ),
            name=f'VEV_{strike}',
            hovertemplate='RV: %{x:.2%}<br>IV: %{y:.2%}<br>Time: %{text}',
            text=merged.index
        ), row=row, col=col)
        
        # Add a y=x reference line (where IV == RV)
        if len(merged) > 0:
            min_val = min(merged[rv_col].min(), merged['IV'].min())
            max_val = max(merged[rv_col].max(), merged['IV'].max())
            
            fig2.add_trace(go.Scatter(
                x=[min_val, max_val],
                y=[min_val, max_val],
                mode='lines',
                line=dict(color='white', dash='dash', width=1),
                showlegend=False,
                hoverinfo='skip'
            ), row=row, col=col)
        
        fig2.update_xaxes(title_text=f"Realized Vol (N={target_w})", tickformat=".1%", row=row, col=col)
        fig2.update_yaxes(title_text="Implied Vol", tickformat=".1%", row=row, col=col)

    fig2.update_layout(
        height=400 * rows,
        title_text="Scatterplot: Implied Volatility vs Realized Volatility",
        template="plotly_dark",
        showlegend=False
    )
    
    fig2.show()
else:
    print("No data to plot.")